# Appendix A — controls, background and detection limits

**Optional. Nothing in the workshop depends on this notebook.**

Notebook 02 uses one number from the control features — `control_frac`, the fraction
of a cell's signal that lands on true negative controls — and moves on. That is the
right level of detail for a first pass.

This appendix is what sits underneath it, and it is worth reading if you are going to
run your own Xenium experiment:

- which non-gene features are actually controls, and which are not
- how to turn the controls into a detection threshold for genes
- why "above background" and "usable" are different questions

Run it after the workshop, or during the exercises if you finish early.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc
import seaborn as sns

sc.settings.verbosity = 1
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA = ROOT / "data"
NAVY, GOLD, CORAL, ICE = "#001158", "#FBAE40", "#F26B43", "#BCD2FF"

# Load the raw object: the non-gene features are still present here.
adata = sc.read_h5ad(DATA / "ovarian_subset.h5ad")
adata.layers["counts"] = adata.X.copy()
x, y = adata.obsm["spatial"].T
print(adata)

## 1. Background: what do the negative controls say?

Two independent estimates of how much of your signal is noise.

If a negative control **probe** fires, chemistry is binding where it should not.
If a negative control **codeword** fires, the decoder is hallucinating.
Splitting them tells you *which* part of the pipeline is misbehaving.

The catch, and the reason this section is longer than you might expect: the matrix
contains several other non-gene feature classes that are **not** negative controls.
Treating them as such is the commonest way to convince yourself a clean run is
broken.

In [ ]:
import scipy.sparse as sp

X = adata.layers["counts"]
tot = np.asarray(X.sum(axis=0)).ravel()      # total counts per feature
is_ctrl = adata.var["control"].to_numpy()

if not is_ctrl.any():
    raise RuntimeError(
        "No control features in this object — see the diagnostic printed above. "
        "The rest of section 1 measures background from the controls, so it "
        "cannot run. Skip to section 2; everything from there on works fine."
    )

# Not every non-gene feature measures background. Only these two are designed
# as negative controls:
#   Negative Control Probe    - a probe against a sequence not in the tissue
#   Negative Control Codeword - a codeword no probe uses
# The others mean different things and are reported separately:
#   Unassigned Codeword  - a valid codeword with no gene assigned to it
#   Genomic Control      - probe binding genomic DNA rather than mRNA
#   Deprecated Codeword  - codewords the pipeline does not use; NOT a control
BACKGROUND_CLASSES = ["Negative Control Probe", "Negative Control Codeword"]

ftypes = (adata.var["feature_types"].astype(str)
          if "feature_types" in adata.var.columns
          else pd.Series("unknown", index=adata.var_names))
is_background = ftypes.isin(BACKGROUND_CLASSES).to_numpy() & is_ctrl
is_other = is_ctrl & ~is_background

if not is_background.any():
    print("No designated negative-control features found; falling back to all")
    print("non-gene features, which will overestimate background.\n")
    is_background = is_ctrl
    is_other = np.zeros_like(is_ctrl)

gene_tot = tot[~is_ctrl]
bg_tot = tot[is_background]

print(f"targeted genes    : {len(gene_tot):>6,} features")
print(f"negative controls : {len(bg_tot):>6,} features   <- background comes from these")
print(f"other non-gene    : {int(is_other.sum()):>6,} features   (separate table below)")

print(f"\nmean counts per targeted gene    : {gene_tot.mean():9.2f}")
print(f"mean counts per negative control : {bg_tot.mean():9.2f}")
print(f"\nbackground rate, mean ratio      : {bg_tot.mean() / gene_tot.mean():9.4f}")
print(f"background rate, median ratio    : "
      f"{np.median(bg_tot) / max(np.median(gene_tot), 1):9.4f}")

### Not every non-gene feature is a control

This distinction matters and is easy to miss. The matrix holds up to five kinds of
non-gene feature, and only two of them measure background:

| Class | What it is | Use for background? |
|---|---|---|
| **Negative Control Probe** | probe against a sequence absent from the tissue | **yes** — measures chemistry |
| **Negative Control Codeword** | a codeword no probe uses | **yes** — measures decoding |
| Unassigned Codeword | valid codeword, no gene assigned | a rough decoding check |
| Genomic Control | probe binding genomic DNA, not mRNA | no — measures sample prep |
| Deprecated Codeword | codewords the pipeline does not use | **no — not a control at all** |

Deprecated codewords are the trap. 10x describes them as codewords not used by the
onboard analysis pipeline: retired from the active panel but still present in the
codebook. They can accumulate enormous counts, and including them in a background
estimate inflates it by orders of magnitude — turning a clean run into an apparent
disaster.

All five are still removed from the count matrix; none is a panel gene. The
distinction is purely about what you use to *estimate background*.

In [ ]:
# Per-class breakdown. Each class fails for a different reason, so whichever
# class is elevated tells you where to look.
rows = []
for cls in sorted(ftypes[is_ctrl].unique()):
    m = (ftypes == cls).to_numpy() & is_ctrl
    rows.append({
        "class": cls,
        "n features": int(m.sum()),
        "total counts": int(tot[m].sum()),
        "median/feature": round(float(np.median(tot[m])), 1),
        "max/feature": int(tot[m].max()),
        "% of all counts": round(100 * tot[m].sum() / tot.sum(), 3),
        "background?": "yes" if cls in BACKGROUND_CLASSES else "no",
    })
display(pd.DataFrame(rows).set_index("class"))

dep = (ftypes == "Deprecated Codeword").to_numpy() & is_ctrl
if dep.any() and tot[dep].sum() > 3 * max(tot[is_background].sum(), 1):
    print("Deprecated codewords carry far more counts than the true negative")
    print("controls here. That is common and is NOT a quality problem — they are")
    print("not controls. Judge this run on the two negative-control rows only.")

**How to read that background rate.** It is roughly "what fraction of an average
gene's signal could be noise". Calibrate against what Xenium actually achieves, not
against scRNA-seq intuition:

| Background rate | Verdict |
|---|---|
| ~0.0001 (1e-4) | a good Prime run — negative controls barely fire at all |
| ~0.001 | still fine |
| ~0.01 | worth investigating; weak genes are becoming unreliable |
| >0.05 | something is wrong with the chemistry or the section |

A typical clean run puts a few dozen counts in total across ~650 negative-control
features, against millions of counts on genes. If your rate is around 1e-4, background
is **not** your limiting problem — statistical power is. That changes what the
threshold below is for.

Compare the mean and median versions. If they disagree a lot, a handful of extreme
features is doing the damage rather than a raised floor.

> **Panel design still bites.** Genes at the bottom of the dynamic range are the ones
> people are most excited about, and the ones you can say least about — but on a clean
> run the reason is *too few counts to estimate anything*, not contamination.

In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 4.2))
bins = np.logspace(0, np.log10(max(tot.max(), 10)), 70)
ax.hist(np.clip(gene_tot, 1, None), bins=bins, color=NAVY, alpha=0.85,
        label=f"targeted genes (n={len(gene_tot):,})")
if is_other.any():
    ax.hist(np.clip(tot[is_other], 1, None), bins=bins, color="0.6", alpha=0.6,
            label=f"other non-gene (n={int(is_other.sum()):,})")
ax.hist(np.clip(bg_tot, 1, None), bins=bins, color=CORAL, alpha=0.95,
        label=f"negative controls (n={len(bg_tot):,})")
ax.set_xscale("log"); ax.set_yscale("log")
ax.set_xlabel("total counts in crop"); ax.set_ylabel("features")
ax.legend(fontsize=8)
ax.set_title("Only the orange distribution measures background")
sns.despine(); plt.show()

floor = np.percentile(bg_tot, 99)
n_below = int((gene_tot < floor).sum())
print(f"99% of negative controls sit below {floor:.0f} counts.")
print(f"{n_below} targeted genes ({100 * n_below / len(gene_tot):.1f}%) fall below "
      f"that — treat them as not measurable in this crop.")

### Reading this figure

Three separate things are visible, and only one of them is a problem.

**The wall at exactly 1 count.** A Prime 5K codebook contains far more codewords than
it has genes, so thousands of unassigned codewords sit in the matrix, each firing
essentially never. Thousands of features with one count each is the *expected*
picture and is good news: the decoder almost never invents a codeword. Do not read the
height of that bar as "lots of background" — it is lots of *features*, each carrying
almost nothing.

**The scattered controls between roughly 2 and 30 counts.** This is the real
background floor. Compare it with where your genes of interest sit.

**The handful of controls out at 10⁴–10⁵ counts.** *This* is the part worth
investigating. A few individual control features carrying more counts than most real
genes is not ordinary background — it is usually one probe cross-hybridising, or one
codeword a single bit away from a highly expressed gene. Those few features also drag
the mean upwards, which is why the summary statistic above can look alarming when the
bulk of the distribution is fine.

The next cell names them.

In [ ]:
# Which non-gene features carry the counts? Almost always a short list.
ctrl_counts = pd.Series(tot[is_ctrl], index=adata.var_names[is_ctrl]).sort_values(ascending=False)
top = ctrl_counts.head(10).to_frame("counts")
top["class"] = ftypes.reindex(top.index)
top["x median gene"] = (top["counts"] / max(np.median(gene_tot), 1)).round(1)
top["is a real control"] = top["class"].isin(BACKGROUND_CLASSES).map({True: "yes", False: "no"})
display(top)

n_ones = int((ctrl_counts == 1).sum())
print(f"{n_ones:,} non-gene features have exactly 1 count (of {len(ctrl_counts):,})")

if (~top["class"].isin(BACKGROUND_CLASSES)).all():
    print("\nNone of the top ten is a negative control — so none of them tells you")
    print("anything about background. Look at the negative-control row of the table")
    print("above instead.")

In [ ]:
# Mean vs median for the two distributions that matter.
summary = pd.DataFrame({
    "targeted genes": [len(gene_tot), gene_tot.mean(), np.median(gene_tot),
                       np.percentile(gene_tot, 95), gene_tot.max()],
    "negative controls": [len(bg_tot), bg_tot.mean(), np.median(bg_tot),
                          np.percentile(bg_tot, 95), bg_tot.max()],
}, index=["n features", "mean", "median", "95th pct", "max"]).round(2)
display(summary)

# Where do specific genes sit relative to the background floor?
floor = np.percentile(bg_tot, 99)
for g in ["EPCAM", "TOP2A", "TRAC", "CXCR4"]:
    if g in adata.var_names:
        v = tot[adata.var_names.get_loc(g)]
        verdict = "below the control floor — do not trust it" if v < floor else \
                  ("close to the floor — cluster means only" if v < 3 * floor
                   else "well above background")
        print(f"  {g:<8} {v:>9,.0f} counts   {verdict}")

### What to do about the outlier controls

They are not a reason to discard the dataset. Two practical responses:

1. **Note them, and check your genes of interest are not among the affected.** One
   cross-hybridising probe says nothing about the rest of the panel.
2. **If an outlier is an *unassigned codeword***, it may be one bit away from a highly
   expressed gene in the codebook. That is decoding bleed, and it means that gene's
   own counts are slightly inflated too.

What would genuinely worry you is the opposite shape: the whole orange distribution
shifted right, overlapping the middle of the blue one. That is a run with systematic
background, and no downstream filtering repairs it.

### Exercise 2.1b
Take the single largest control feature. How many targeted genes in this panel have
*fewer* counts than it? Those genes are, in this section, less reliably measured than
a feature that is supposed to measure nothing.

In [ ]:
# your code here

### Two different thresholds, and you need both

"Which genes can I use?" is really two questions.

**1. Is this gene above background?** The negative controls answer this, because they
are features that measure nothing. For a candidate threshold T: the fraction of
controls reaching T estimates the chance a pure-noise feature reaches T; multiply by
the number of genes for expected false positives; divide by genes actually above T.
That is an empirical **false discovery rate**, with the controls playing the role of a
permutation null.

**2. Is this gene usable?** A different question entirely. On a clean Prime run the
FDR threshold lands at two or three counts — statistically defensible and practically
useless, because a gene with three counts across twenty thousand cells cannot support
a mean, a fold change or a spatial statistic.

So compute both. Question 1 protects you from noise. Question 2 protects you from
underpowered claims, and on a clean run **it is the binding constraint**.

In [ ]:
def detection_fdr(gene_counts, control_counts, thresholds=None):
    """Empirical FDR for calling a gene 'detected', using controls as the null."""
    if thresholds is None:
        hi = max(int(np.percentile(control_counts, 100)) * 3, 20)
        thresholds = np.unique(np.round(np.logspace(0, np.log10(hi), 40)).astype(int))

    rows = []
    for T in thresholds:
        p = float((control_counts >= T).mean())     # P(noise feature reaches T)
        expected_false = p * len(gene_counts)
        kept = int((gene_counts >= T).sum())
        rows.append({
            "threshold": int(T),
            "controls >= T": int((control_counts >= T).sum()),
            "genes kept": kept,
            "expected false": round(expected_false, 1),
            "FDR": round(min(expected_false / max(kept, 1), 1.0), 4),
        })
    return pd.DataFrame(rows)


fdr = detection_fdr(gene_tot, bg_tot)
display(fdr.head(15))

resolution = 1 / len(bg_tot)
print(f"\nYou have {len(bg_tot)} negative controls, so the smallest non-zero noise")
print(f"probability you can measure is 1/{len(bg_tot)} = {resolution:.4f}, i.e. about")
print(f"{resolution * len(gene_tot):.0f} expected false positives. You cannot ask for a")
print("finer FDR than that — the controls simply do not have the resolution.")

In [ ]:
TARGET_FDR = 0.05        # <-- CHANGE THIS (try 0.10, 0.05, 0.01)

ok = fdr[fdr["FDR"] <= TARGET_FDR]
if len(ok) == 0:
    raise RuntimeError(
        f"No threshold reaches FDR <= {TARGET_FDR} with {len(bg_tot)} controls. "
        "Relax the target, or accept that this panel cannot support it."
    )
MIN_GENE_COUNTS = int(ok["threshold"].iloc[0])
kept = int(ok["genes kept"].iloc[0])

print(f"threshold for FDR <= {TARGET_FDR}: {MIN_GENE_COUNTS} counts in the crop")
print(f"keeps {kept:,} of {len(gene_tot):,} genes "
      f"({100 * kept / len(gene_tot):.1f}%)")

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(fdr["threshold"], fdr["FDR"], color=NAVY, lw=2, label="estimated FDR")
ax.axhline(TARGET_FDR, color=GOLD, lw=2, ls="--", label=f"target {TARGET_FDR}")
ax.axvline(MIN_GENE_COUNTS, color=CORAL, lw=2, label=f"threshold = {MIN_GENE_COUNTS}")
ax.set_xscale("log"); ax.set_xlabel("threshold (total counts in crop)")
ax.set_ylabel("estimated FDR"); ax.legend(fontsize=8)

ax2 = ax.twinx()
ax2.plot(fdr["threshold"], fdr["genes kept"], color="0.6", lw=1.4, ls=":")
ax2.set_ylabel("genes kept", color="0.5")
sns.despine(right=False); plt.show()

### Flag the genes — do not delete them

Two reasons to add a column rather than subset the object.

**Deleting changes the arithmetic.** `total_counts` per cell is computed over whatever
genes are present, so removing genes shifts every normalisation and every QC number
you already looked at.

**Low total counts does not mean noise.** A marker of a rare population — the ciliated
cells in this section are a few hundred cells — has low counts *in total* while being
completely real and highly expressed in the cells that have it. A total-count
threshold is biased against exactly the rare biology you might care about most.

So flag, then decide per analysis: use all genes for clustering, where a weak gene
contributes little either way, and restrict to the reliable set when you report
differential expression or interpret a single gene.

In [ ]:
adata.var["above_background"] = False
adata.var.loc[~adata.var["control"], "above_background"] = (
    tot[~adata.var["control"].to_numpy()] >= MIN_GENE_COUNTS
)
adata.uns["detection_threshold"] = {
    "min_counts": MIN_GENE_COUNTS,
    "target_fdr": TARGET_FDR,
    "n_negative_controls": int(len(bg_tot)),
}
print(adata.var.loc[~adata.var["control"], "above_background"].value_counts())

# Rescue check: is a gene below the threshold nevertheless concentrated in a few
# cells? That is the signature of a rare cell type, not of noise.
below = (~adata.var["above_background"]) & (~adata.var["control"])
if below.any():
    Xb = adata.layers["counts"][:, below.to_numpy()]
    n_cells_pos = np.asarray((Xb > 0).sum(axis=0)).ravel()
    mx = Xb.max(axis=0)
    max_in_cell = np.asarray(mx.todense() if sp.issparse(mx) else mx).ravel()
    rescue = pd.DataFrame({
        "total": tot[below.to_numpy()],
        "cells detected": n_cells_pos,
        "max in one cell": max_in_cell,
    }, index=adata.var_names[below.to_numpy()])
    # concentrated = few cells, but several copies in those cells
    rescue["concentration"] = rescue["max in one cell"] / rescue["total"].clip(lower=1)
    candidates = rescue.query("total >= 5").sort_values("concentration", ascending=False)
    print("\nBelow threshold, but concentrated in few cells — check these by hand:")
    display(candidates.head(10))

### The usability threshold

Total counts are the wrong currency for this, because a gene concentrated in a rare
population has few total counts and is perfectly usable *within* that population. The
right question is **in how many cells is it detected at all** — that is what sets
whether you can compute anything about it.

Rules of thumb, to adapt rather than obey:

- **detected in ≥ 3% of cells** — safe for per-cluster means and differential expression
- **detected in ≥ 30 cells** — the floor for saying anything at all, and only about
  the population those cells belong to
- **fewer than that** — you can report the transcripts as observed, but not a statistic

In [ ]:
n_cells_detected = np.asarray((adata.layers["counts"] > 0).sum(axis=0)).ravel()
frac_cells = n_cells_detected / adata.n_obs
adata.var["n_cells_detected"] = n_cells_detected
adata.var["frac_cells_detected"] = frac_cells

MIN_FRAC_CELLS = 0.03        # <-- CHANGE THIS (try 0.01, 0.03, 0.10)
MIN_CELLS = 30

genes_only = ~adata.var["control"].to_numpy()
usable = genes_only & (frac_cells >= MIN_FRAC_CELLS) & (n_cells_detected >= MIN_CELLS)
adata.var["usable"] = usable

n_genes_total = int(genes_only.sum())
print(f"above background (FDR)          : {int(adata.var['above_background'].sum()):>5,} / {n_genes_total:,}")
print(f"usable (>= {100 * MIN_FRAC_CELLS:.0f}% of cells)         : {int(usable.sum()):>5,} / {n_genes_total:,}")
print(f"both                            : "
      f"{int((usable & adata.var['above_background'].to_numpy()).sum()):>5,}")

fig, ax = plt.subplots(figsize=(6.5, 4.5))
ax.scatter(np.clip(tot[genes_only], 1, None), np.clip(frac_cells[genes_only], 1e-5, None),
           s=3, c=NAVY, alpha=0.35, linewidths=0, rasterized=True)
ax.axvline(MIN_GENE_COUNTS, color=CORAL, lw=2,
           label=f"above background: {MIN_GENE_COUNTS} counts")
ax.axhline(MIN_FRAC_CELLS, color=GOLD, lw=2,
           label=f"usable: {100 * MIN_FRAC_CELLS:.0f}% of cells")
ax.set_xscale("log"); ax.set_yscale("log")
ax.set_xlabel("total counts in crop"); ax.set_ylabel("fraction of cells detected in")
ax.legend(fontsize=8, loc="lower right")
ax.set_title("the two thresholds do different work")
sns.despine(); plt.show()

### Read the scatter

The two lines rarely coincide, and where they diverge tells you something.

- **Right of the orange line, below the gold** — above background, but detected in too
  few cells to analyse. Either a rare-cell-type marker (interesting, and analysable
  *within* that population) or a gene expressed at a trickle everywhere.
- **Above the gold line, left of the orange** — detected broadly but at very low counts.
  On a clean run this is unusual; if you see many, look again at the background.
- The cloud running diagonally is the normal relationship: more counts, more cells.

**Which do you report?** Say both in the methods. "Genes were required to exceed the
negative-control null at FDR < 0.05 (≥ N counts) and to be detected in ≥ 3% of cells"
is one sentence, and it tells a reader exactly what your gene space was.

### Exercise 2.1d
Find the genes that pass the background test but fail the usability test. Are any of
them markers you would have wanted? Then check whether they are concentrated in one
region of the tissue — a gene detected in 1% of cells that are all in one place is a
very different object from one detected in 1% of cells scattered at random.

In [ ]:
# your code here

### The cross-check worth remembering

There is a second, independent way to ask whether a weak gene is real, and it is only
available because this is spatial data: **a real gene is spatially structured; noise
is not.**

In notebook 04 you compute Moran's I for every gene. A gene sitting just below your
count threshold but with clearly non-random spatial autocorrelation is almost
certainly real — noise does not form patches. A gene above the threshold with Moran's
I near zero deserves more suspicion than its count suggests.

That check has no equivalent in dissociated data, and it is a better arbiter than any
count cutoff.

### Exercise 2.1c
Set `TARGET_FDR` to 0.10 and to 0.01. How many genes does each keep? Then find one
gene that is included at 0.10 but excluded at 0.01, and decide from its spatial
pattern which call you believe.

### Exercise 2.1
Pick three genes you would actually want to use in an ovarian tumour study
(e.g. `MKI67`, `CD8A`, `PDCD1`). Where does each sit relative to the control
distribution in this crop? Would you trust a per-cell measurement of it, a
per-cluster mean, or neither?

In [ ]:
# your code here

---

## Missing markers and deprecated codewords

If a workhorse marker like `COL1A1` or `TAGLN` turns out to be absent from a
5,000-gene pan-tissue panel, that is worth two minutes. One possible reason connects
directly to the control classes above: the probe was retired. Retired probes still
detect real transcripts, which is why deprecated codewords can carry enormous counts —
and the test for whether that is what happened is spatial.

### When a marker you expected is not on the panel

Some absences are unsurprising. But if a workhorse marker like `COL1A1`, `TAGLN` or
`ACTA2` is missing from a 5,000-gene pan-tissue panel, that is worth two minutes,
because there are three quite different explanations:

1. **It really is not on the panel.** Check `gene_panel.json` from the run — that file
   is authoritative, not the matrix.
2. **It is there under a different symbol.** Aliases and older nomenclature do occur.
   The `find_gene` helper above already tries a case-insensitive match.
3. **Its probe was deprecated.** This is the interesting one. Deprecated codewords are
   probes retired from the active panel — and they still detect real transcripts. A
   retired `COL1A1` probe would produce exactly what notebook 02 found: a deprecated
   codeword carrying hundreds of thousands of counts, concentrated in fibrous stroma.

Explanation 3 is testable, and the test is spatial.

In [ ]:
# Do the big deprecated codewords behave like the missing markers would?
# Load the raw object so the non-gene features are still present.
raw = sc.read_h5ad(DATA / "ovarian_subset.h5ad")

ft = raw.var["feature_types"].astype(str) if "feature_types" in raw.var.columns else None
if ft is None:
    print("no feature_types column — cannot run this check")
else:
    dep = (ft == "Deprecated Codeword").to_numpy()
    dep_tot = np.asarray(raw[:, dep].X.sum(axis=0)).ravel()
    top_dep = pd.Series(dep_tot, index=raw.var_names[dep]).nlargest(3)
    print("largest deprecated codewords:")
    print(top_dep.to_string())

    # Where are they? If a retired collagen probe, expect fibrous stroma:
    # a broad band, not a compact nest.
    xs, ys = raw.obsm["spatial"].T
    fig, axes = plt.subplots(1, len(top_dep), figsize=(4.6 * len(top_dep), 4.6))
    for ax, name in zip(np.atleast_1d(axes), top_dep.index):
        v = np.asarray(raw[:, name].X.todense()).ravel()
        order = np.argsort(v)
        pc = ax.scatter(xs[order], ys[order], c=v[order], s=1.1, cmap="magma",
                        vmax=np.percentile(v, 99.5), linewidths=0, rasterized=True)
        plt.colorbar(pc, ax=ax, fraction=0.046)
        ax.set_aspect("equal"); ax.invert_yaxis(); ax.set_xticks([]); ax.set_yticks([])
        ax.set_title(f"{name}\n{int(v.sum()):,} counts", fontsize=9)
    plt.tight_layout(); plt.show()

### How to read those maps

Compare each deprecated codeword against the tissue compartments you already know.

- **Follows the fibrous stroma** — consistent with a retired collagen or matrix probe.
  Compare it directly with `DCN` or `LUM`, which are on the panel.
- **Follows the tumour nests** — a retired epithelial or tumour-associated probe.
- **Diffuse, no structure** — genuine decoding noise, and the counts are meaningless.

**This matters beyond curiosity.** If a large deprecated codeword tracks a compartment,
then the panel is systematically under-reporting that compartment's transcripts: the
signal exists in the run but is not attributed to any gene. Your cell-type proportions
and your per-cell totals for those cells are both affected — which is exactly the
region notebook 02 found being discarded.

**What to do.** You cannot recover the gene identity from the matrix; the mapping from
deprecated codeword to retired target lives with 10x. Two practical responses: state
in the methods that the panel version retired probes whose signal is unattributed, and
do not use raw transcript totals as a proxy for transcriptional activity in the
affected compartment.

### Exercise 3.4
Correlate the largest deprecated codeword with `DCN`, `EPCAM` and `PTPRC` across cells.
Which compartment does it belong to? Then check whether the cells expressing it are
the same cells your QC filter discarded in notebook 02.

In [ ]:
# your code here